# Passing callback to the `synthetmic.DiagramGenerator.fit` method

In this example, we will show how to pass a callback function to the
`synthetmic.DiagramGenerator.fit` method, particularly,
`synthetmic.LaguerreDiagramGenerator.fit` (the same approach
can be used for `synthetmic.VoronoiDiagramGenerator.fit`).

A callback function can be used to track the progress of the algorithm
implemented in the `synthetmic.DiagramGenerator.fit` method. In this example, we will use
the popular `tqdm` package to track the progress of our microstructure
generation.

We will create a 3D Laguerre diagram with all grains of equal volume.


We first install `synthetmic` and `tqdm`.

In [ ]:
!pip install synthetmic tqdm matplotlib

Next, we import the necessary classes and functions.

In [ ]:
from synthetmic import LaguerreDiagramGenerator, LaguerreEvent
from synthetmic.data.toy import create_data_with_constant_volumes
import matplotlib.pyplot as plt
from tqdm import tqdm

Below, we create a unit cube with 10,000 grains.
All grains will have volume $1/10000$ cubic units (up to 1% tolerance).

In [ ]:
config = create_data_with_constant_volumes(
    space_dim=3,
    n_grains=10_000,
    is_periodic=False,
    random_state=42
)
config

We create a callback function to monitor the progress of the `fit` method.

In [ ]:
N_ITER = 30

diagram = LaguerreDiagramGenerator(
    tol=1.0,
    n_iter=N_ITER,
    damp_param=1.0,
)

In [ ]:
pbar = tqdm(total=N_ITER, desc="Running")
def callback(e: LaguerreEvent) -> None:
        pbar.n = e.iteration
        pbar.set_postfix(
                {
                        "mean_percentage_error": e.mean_percentage_error,
                        "max_percentage_error": e.max_percentage_error,
                },
                refresh=True
        )
        pbar.refresh()
        
diagram.fit(config, callback=callback)
pbar.close()

Here is another example of a callback function, where at each iteration we print out the mean and maximum percentage error of the grain volumes:  

In [ ]:
def callback(e: LaguerreEvent) -> None:
    print(f"itr: {e.iteration}, mean_p_err: {e.mean_percentage_error}, max_p_err: {e.max_percentage_error}")

diagram.fit(config, callback=callback)


Callback is not limited to showing iteration progress but also can be used to accumulate iteration metrics. The accumulated metrics can  then be processed further, for instance, plotting the metrics against iterations.

We domonstrate this below.

In [ ]:
history = {
    "Iteration": [],
    "Mean volume percentage error": [],
    "Max volume percentage error": [],
    "Centroid error": []
}
def callback(e: LaguerreEvent) -> None:
    history["Iteration"].append(e.iteration)
    history["Mean volume percentage error"].append(e.mean_percentage_error)
    history["Max volume percentage error"].append(e.max_percentage_error)
    history["Centroid error"].append(e.centroid_error_norm)

diagram.fit(config, callback=callback)

In [ ]:
_, ax = plt.subplots()
for err in ("Mean volume percentage error", "Max volume percentage error"):
    ax.plot(
        history["Iteration"],
        history[err],
        linewidth=2,
        label=err,
        marker="o",
        markerfacecolor='white',
    )

ax.set_yscale('log')
ax.set_xlabel("Iteration")
ax.set_ylabel("Error (%)")
ax.legend()


In [ ]:
_, ax = plt.subplots()
ax.plot(
        history["Iteration"],
        history["Centroid error"],
        linewidth=2,
        marker="o",
        markerfacecolor='white',
    )

ax.set_yscale('log')
ax.set_xlabel("Iteration")
ax.set_ylabel("Centroid error")